In [ ]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
import corvinus_tardis_data as ctd
import seaborn as sns
import matplotlib.pyplot as plt
# this is a special fix for tardis_dev not liking running from a notebook. it prefers running as a script


In [ ]:
# instead of using the notebook cell, execute in subshell (because asyncio)
! python -m corvinus_tardis_data
! ls -l datasets

Let's read the dataset and organize the columns

In [ ]:
options = pd.read_csv('datasets/binance-european-options_trades_2025-05-01_OPTIONS.csv.gz')
options['timestamp'] = pd.to_datetime(options['timestamp'], unit='us')
options['timestamp'] = pd.to_datetime(options['timestamp'], unit='us')

options = options.groupby(['exchange','symbol','timestamp','side']).price.mean()
options = options.reset_index()

options['base'] = options.symbol.str.split('-').str[0]
options['expiration'] = options.symbol.str.split('-').str[1]
options['strike'] = options.symbol.str.split('-').str[2].astype(float)
options['pc'] = options.symbol.str.split('-').str[3]

options


Take a particular symbol, and see when it traded

In [ ]:
sample = options.loc[options.symbol=='ETH-250501-1775-P'].copy()
mint, maxt = min(sample.timestamp), max(sample.timestamp)
time_range = pd.date_range(mint, maxt, freq="5 min")

sample['traded'] = 1
sample['num_trades'] = sample.traded.cumsum()

on_grid = sample.set_index('timestamp').reindex(time_range, method='ffill')[['price','num_trades']].reset_index()
on_grid['event'] = on_grid.num_trades != on_grid.num_trades.shift()
on_grid.loc[~on_grid.event, 'price'] = np.nan

on_grid.price.plot()
display(on_grid.head())
display(sample.head())

Redo the same but now put the procedure into a function

In [ ]:
sample = options.loc[options.symbol=='ETH-250501-1775-P'].copy()
mint, maxt = min(sample.timestamp), max(sample.timestamp)
time_range = pd.date_range(mint, maxt, freq="5 min").rename('timestamp')

def one_symbol(g):
    g = g.copy()
    g['traded'] = 1
    g['num_trades'] = g.traded.cumsum()
    on_grid = g.set_index('timestamp').reindex(time_range, method='ffill')[['price','num_trades']].reset_index()
    g['event'] = g.num_trades!= g.num_trades.shift()
    on_grid['event'] = on_grid.num_trades != on_grid.num_trades.shift()
    on_grid.loc[~on_grid.event, 'price'] = np.nan
    return on_grid[['timestamp','price']]

one_symbol(sample).set_index('timestamp').price.plot()


Now do all of this for each of the symbols in the file

In [ ]:


sample = options.copy()
#display(sample)
#display(sample.groupby('symbol')[['timestamp','price']].first())
on_grid = sample.groupby('symbol')[['timestamp','price']].apply(one_symbol)

sns.lineplot(data=on_grid.reset_index(), x='timestamp', hue='symbol',y='price')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.legend().remove()
on_grid

In [ ]:

sample = options.loc[options.base == 'BTC']
#display(sample)
on_grid = sample.groupby('symbol')[['timestamp','price']].apply(one_symbol).reset_index()
#display(on_grid.head())
df = on_grid.set_index(['timestamp','symbol']).unstack('timestamp').price
#display(df)
toplot = (~df.isna()).astype(float).describe().T
#display(toplot)
toplot['mean'].plot()